<a href="https://colab.research.google.com/github/EkaterinaLavlinskaya/ai-agents-for-local-use/blob/main/AI_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq && apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 97 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (613 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

<Popen: returncode: None args: ['ollama', 'serve']>

In [ ]:
!ollama pull qwen2.5:1.5b

In [ ]:
%pip install -U langchain langchain-core langchain-ollama langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.4/170.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.0/869.0 kB 52.6 MB/s eta 0:00:00
  Attempting uninstall: langsmith
    Found existing installation: langsmith 0.14.2
    Uninstalling langsmith-0.14.2:
      Successfully uninstalled langsmith-0.14.2
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.6
    Uninstalling langchain-core-1.6.6:
      Successfully uninstalled langchain-core-1.6.6
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.4.5
    Uninstalling langgraph-sdk-0.4.5:
      Successfully uninstalled langgr

In [ ]:
with open("Modelfile", "w") as f:
    f.write("FROM qwen2.5:7b\n")

In [ ]:
!ollama create my-qwen -f Modelfile

In [ ]:
import subprocess
import time

# Запускаем сервер Ollama в фоне
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Даём серверу время запуститься
time.sleep(3)

# Проверяем, что сервер отвечает (опционально)
!curl http://localhost:11434/api/tags

{"models":[{"name":"my-qwen:latest","model":"my-qwen:latest","modified_at":"2026-10-19T08:40:23.934362412Z","size":4683086960,"digest":"65199436211653c72ad95f3eae40a20841d0d94fa86d7eb97f6e14eb64b1f096","details":{"parent_model":"qwen2.5:7b","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":3584,"runner":"llamacpp"},"capabilities":["completion","tools"]},{"name":"qwen2.5:7b","model":"qwen2.5:7b","modified_at":"2026-10-19T08:40:23.686345188Z","size":4683087332,"digest":"845dbda0ea48ed749caafd9e6037047aa19acfcfd82e704d7ca97d631a0b697e","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":3584,"runner":"ggml"},"capabilities":["completion","tools"]},{"name":"qwen2.5:1.5b","model":"qwen2.5:1.5b","modified_at":"2026-10-19T08:39:28.586581627Z","size":986061892,"digest":"6

In [ ]:
from datetime import datetime

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

CHAT_MODEL = "qwen2.5:1.5b"   # Чат-модель Ollama. Должна поддерживать вызов инструментов.

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools for getting the current time and counting words in text. "
    "Use tools when the user's request needs one. "
    "If the question doesn't need a tool, answer directly. "
    "If a tool returns an error, explain the error plainly."
)

# ----- Tools -----
@tool
def current_time() -> str:
    """Возвращает текущую локальную дату и время.
    Используйте этот инструмент, когда пользователь спрашивает, который час или день.
    """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def word_count(text: str) -> int:
    """Подсчитывает количество слов в тексте.
    Используйте этот метод, когда пользователь спрашивает, сколько слов в тексте,
    или просит вас подсчитать количество слов в чем-то, чем он поделился.
    Возвращает количество слов в виде целого числа.
    """
    return len(text.split())

TOOLS = [current_time, word_count]

# ----- Agent -----
def build_agent():
    model = ChatOllama(model="my-qwen", temperature=0.7, repeat_penalty=1.1)

    # InMemorySaver хранит историю диалога в памяти, индексируя ее по идентификатору потока.
    # После завершения процесса история удаляется из-за кратковременной памяти.
    checkpointer = InMemorySaver()

    return create_agent(
        model=model,
        tools=TOOLS,
        system_prompt=SYSTEM_PROMPT,
        checkpointer=checkpointer,
    )

def main():
    agent = build_agent()

    # Идентификатор потока сообщает чекпойнтеру, какой диалог нужно загрузить и сохранить.
    config = {"configurable": {"thread_id": "thread"}}

    print("Ready! Ask the agent something. It remembers the conversation.\n")

    # Отследим, сколько сообщений было до этого хода, чтобы выделить
    # только новые (вызовы инструментов + окончательный ответ) из возвращенного состояния.
    prev_message_count = 0

    while True:
        question = input("You: ").strip()
        if not question or question.lower() == "exit":
            break

        result = agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            config=config,
        )

        # Смотрим только на сообщения, добавленные в этот ход, а не на всю историю.
        new_messages = result["messages"][prev_message_count:]

        # Выводим все вызовы инструментов, сделанные в этот ход.
        for msg in new_messages:
            tool_calls = getattr(msg, "tool_calls", None)
            if tool_calls:
                for call in tool_calls:
                    print(f"[tool call] {call['name']}({call['args']})")

        print(f"\nAnswer: {result['messages'][-1].content}\n")

        # Обновляем счетчик для следующего хода.
        prev_message_count = len(result["messages"])

if __name__ == "__main__":
    main()

Ready! Ask the agent something. It remembers the conversation.

You: Привет

Answer: Привет! Как я могу помочь вам сегодня?

You: проверяю как ты работаешь

Answer: Хорошо, расскажите мне о вашем тесте, или если у вас есть вопросы, я постараюсь ответить на них.

You: смотрю какие промнты тебе больше подходят .Какой язык лучше работает

Answer: Чтобы дать точный ответ на ваш вопрос, нужно больше информации. Вы имеете в виду лучший язык программирования для создания пользовательских ботов или для выполнения определенного типа задач? Например, Python часто используется из-за своей читаемости и обширной библиотекарии, а JavaScript хорош для веб-приложений. Пожалуйста, уточните ваш вопрос!



KeyboardInterrupt: Interrupted by user